<a href="https://colab.research.google.com/github/nikhildhavale/pythonLearning/blob/main/lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RND = 42
np.random.seed(RND)
tf.random.set_seed(RND)

print("TensorFlow:", tf.__version__)
print("Libraries loaded.")
DATA_PATH = "shakespeare.txt"

import os
if not os.path.exists(DATA_PATH):
    try:
        from urllib.request import urlretrieve
        urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", DATA_PATH)
        print("Downloaded shakespeare.txt from fallback URL.")
    except Exception as e:
        raise FileNotFoundError(f"{DATA_PATH} not found. In Colab, upload the file or check the URL. {e}")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = np.array(chars)

data = np.array([char2idx[c] for c in text], dtype=np.int32)

print("Vocab size:", vocab_size)
print("Text length (chars):", len(text))
print("First 80 chars:", repr(text[:80]))

seq_length = 40
step = 1

sentences = []
next_chars = []
for i in range(0, len(data) - seq_length, step):
    sentences.append(data[i : i + seq_length])
    next_chars.append(data[i + seq_length])

X = np.array(sentences, dtype=np.int32)
y = np.array(next_chars, dtype=np.int32)

split = int(0.9 * len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print("X_train:", X_train.shape, "  y_train:", y_train.shape)
print("X_val:", X_val.shape, "  y_val:", y_val.shape)
model_rnn = keras.Sequential([
    layers.Input(shape=(seq_length,)),
    layers.Embedding(vocab_size, 64),
    layers.SimpleRNN(128),
    layers.Dense(vocab_size, activation="softmax"),
])
model_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_rnn = model_rnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=512,
    verbose=1,
)

val_loss_rnn = history_rnn.history["val_loss"][-1]
val_acc_rnn = history_rnn.history["val_accuracy"][-1]
print("SimpleRNN final val loss:", round(val_loss_rnn, 4), "  val accuracy:", round(val_acc_rnn, 4))
model_lstm = keras.Sequential([
    layers.Input(shape=(seq_length,)),
    layers.Embedding(vocab_size, 64),
    layers.LSTM(128),
    layers.Dense(vocab_size, activation="softmax"),
])
model_lstm.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_lstm = model_lstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=512,
    verbose=1,
)

val_loss_lstm = history_lstm.history["val_loss"][-1]
val_acc_lstm = history_lstm.history["val_accuracy"][-1]
print("LSTM final val loss:", round(val_loss_lstm, 4), "  val accuracy:", round(val_acc_lstm, 4))
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(history_rnn.history["val_loss"], label="SimpleRNN val loss")
plt.plot(history_lstm.history["val_loss"], label="LSTM val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Validation loss: SimpleRNN vs LSTM")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Perplexity (lower is better):")
print("  SimpleRNN:", round(np.exp(val_loss_rnn), 2))
print("  LSTM:", round(np.exp(val_loss_lstm), 2))
def generate_text(model, seed, char2idx, idx2char, seq_length, num_chars, temperature=0.8):
    generated = list(seed)
    for _ in range(num_chars):
        if len(generated) < seq_length:
            pad = [char2idx.get(" ", 0)] * (seq_length - len(generated))
            x = np.array([pad + [char2idx.get(c, 0) for c in generated]], dtype=np.int32)
        else:
            x = np.array([[char2idx.get(c, 0) for c in generated[-seq_length:]]], dtype=np.int32)
        preds = model.predict(x, verbose=0)[0]
        preds = np.asarray(preds).astype("float64")
        preds = np.log(preds + 1e-8) / temperature
        exp_preds = np.exp(preds)
        preds = exp_preds / exp_preds.sum()
        next_idx = np.random.choice(len(preds), p=preds)
        generated.append(idx2char[next_idx])
    return "".join(generated)

seed = "The king"
num_chars = 200
temperature = 0.8

print("Seed:", repr(seed))
print("Number of characters to generate:", num_chars)
print("Temperature:", temperature)
print()
print("--- SimpleRNN generated ---")
out_rnn = generate_text(model_rnn, seed, char2idx, idx2char, seq_length, num_chars, temperature)
print(out_rnn)
print()
print("--- LSTM generated ---")
out_lstm = generate_text(model_lstm, seed, char2idx, idx2char, seq_length, num_chars, temperature)
print(out_lstm)

TensorFlow: 2.19.0
Libraries loaded.
Downloaded shakespeare.txt from fallback URL.
Vocab size: 65
Text length (chars): 1115394
First 80 chars: 'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.'
X_train: (1003818, 40)   y_train: (1003818,)
X_val: (111536, 40)   y_val: (111536,)
Epoch 1/10
  25/1961 ━━━━━━━━━━━━━━━━━━━━ 3:31 109ms/step - accuracy: 0.0758 - loss: 3.8340